# Lesson 14a: Generative Models — Theory

Every earlier lesson built a network that maps an input to something else:
a class, a next token, a translated sentence. This lesson builds networks
that instead learn a distribution $p(x)$ over the data itself, so new,
never-seen examples can be sampled from it. Three families dominate modern
generative modelling — the **VAE**, **diffusion models**, and **GANs** — and
they differ in exactly what quantity they optimise to get there. This
notebook derives the first two from first principles and states the third's
objective for comparison, using MNIST digits throughout as the concrete
data.

## Introduction

A generative model's job is to make $p_\theta(x) \approx p_{\text{data}}(x)$
close enough that sampling from $p_\theta$ looks like sampling from the real
data. This is a fundamentally harder target than classification: a
classifier only needs $p(y \mid x)$ to be right on average across a decision
boundary, while a generative model needs enough of the *entire* joint shape
of $x$-space to produce a convincing, novel sample from anywhere in it. Every
family in this notebook — autoencoders (which fall short), VAEs, and
diffusion — is best understood as successive fixes to a single problem:
what latent representation makes sampling actually work?

## Setup

In [ ]:
# Fixed seeds: every stochastic step in this notebook (weight init, minibatch
# order, noise sampling) is reproducible.
import numpy as np
import torch

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

import torchvision
from torchvision import transforms
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (5, 4)
print("numpy:", np.__version__)
print("torch:", torch.__version__)

device = torch.device("cpu")
print("using device:", device)

In [ ]:
mnist = torchvision.datasets.MNIST(root="data", train=True, download=True,
                                    transform=transforms.ToTensor())

N = 800
rng = np.random.default_rng(SEED)
idx = rng.permutation(len(mnist))[:N]
images = np.stack([np.asarray(mnist[i][0]).reshape(-1) for i in idx]).astype(np.float64)  # (N, 784)
X = images.T  # (784, N) -- features-by-examples, matching 2a's convention

fig, axes = plt.subplots(1, 8, figsize=(13, 2))
for ax, i in zip(axes, range(8)):
    ax.imshow(X[:, i].reshape(28, 28), cmap="gray")
    ax.axis("off")
plt.suptitle("MNIST training samples")
plt.show()
print("X:", X.shape, " range:", X.min(), "-", X.max())

## Autoencoders

An **autoencoder** learns an encoder $f_\phi: x \to z$ into a low-dimensional
bottleneck and a decoder $g_\theta: z \to \hat x$, trained end-to-end to
minimise reconstruction error $\lVert x - g_\theta(f_\phi(x)) \rVert^2$ — no
generative target anywhere in that loss. Both networks below are
hand-derived and trained with plain NumPy gradient descent, in the same
forward/backward style 2a used for the MLP.

In [ ]:
D, H = 784, 32  # input dim, bottleneck dim


def init_ae(seed=SEED):
    g = np.random.default_rng(seed)
    return {
        "W1": g.normal(0, 0.05, (H, D)), "b1": np.zeros((H, 1)),
        "W2": g.normal(0, 0.05, (D, H)), "b2": np.zeros((D, 1)),
    }


def ae_forward(p, X):
    z1 = p["W1"] @ X + p["b1"]
    h = np.tanh(z1)                    # bottleneck activation, bounded in (-1, 1)
    Xhat = p["W2"] @ h + p["b2"]        # linear output (reconstructing [0, 1] pixel intensities)
    return h, Xhat


def ae_backward(p, X, h, Xhat):
    n = X.shape[1]
    dXhat = 2.0 * (Xhat - X) / (D * n)          # d(mean-squared-error)/d(Xhat)
    dW2 = dXhat @ h.T
    db2 = dXhat.sum(axis=1, keepdims=True)
    dh = p["W2"].T @ dXhat
    dz1 = dh * (1.0 - h ** 2)                    # tanh'(z1) = 1 - tanh(z1)^2
    dW1 = dz1 @ X.T
    db1 = dz1.sum(axis=1, keepdims=True)
    return {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2}


def ae_loss(X, Xhat):
    return float(np.mean((Xhat - X) ** 2))


# Numerical gradient check at a handful of parameters, before trusting the
# analytic backward pass for training.
params = init_ae()
h, Xhat = ae_forward(params, X[:, :16])
grads = ae_backward(params, X[:, :16], h, Xhat)
eps = 1e-5
max_rel_err = 0.0
g = np.random.default_rng(1)
for name in ["W1", "b1", "W2", "b2"]:
    for _ in range(5):
        idx_flat = tuple(g.integers(0, s) for s in params[name].shape)
        orig = params[name][idx_flat]
        params[name][idx_flat] = orig + eps
        _, Xhat_p = ae_forward(params, X[:, :16])
        loss_p = ae_loss(X[:, :16], Xhat_p)
        params[name][idx_flat] = orig - eps
        _, Xhat_m = ae_forward(params, X[:, :16])
        loss_m = ae_loss(X[:, :16], Xhat_m)
        params[name][idx_flat] = orig
        numeric = (loss_p - loss_m) / (2 * eps)
        analytic = grads[name][idx_flat]
        rel_err = abs(numeric - analytic) / max(abs(numeric), abs(analytic), 1e-8)
        max_rel_err = max(max_rel_err, rel_err)
print(f"max relative error, analytic vs. numerical gradient: {max_rel_err:.2e}")
assert max_rel_err < 1e-3, "hand-derived autoencoder gradients disagree with finite differences" 

With the backward pass verified, gradient descent trains the reconstruction
loss down directly.

In [ ]:
def train_ae(p, X, lr=0.5, epochs=300):
    losses = []
    for _ in range(epochs):
        h, Xhat = ae_forward(p, X)
        losses.append(ae_loss(X, Xhat))
        grads = ae_backward(p, X, h, Xhat)
        for name in p:
            p[name] -= lr * grads[name]
    return losses


ae_params = init_ae()
ae_losses = train_ae(ae_params, X)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(ae_losses)
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("reconstruction MSE"); axes[0].set_title("Training loss")

_, Xhat = ae_forward(ae_params, X[:, :8])
for i in range(8):
    ax = axes[1].inset_axes([i / 8, 0.5, 1 / 8, 0.5])
    ax.imshow(X[:, i].reshape(28, 28), cmap="gray"); ax.axis("off")
    ax2 = axes[1].inset_axes([i / 8, 0.0, 1 / 8, 0.5])
    ax2.imshow(Xhat[:, i].reshape(28, 28), cmap="gray"); ax2.axis("off")
axes[1].axis("off")
axes[1].set_title("original (top) vs. reconstruction (bottom)")
plt.tight_layout(); plt.show()
print(f"final reconstruction MSE: {ae_losses[-1]:.4f}")

Reconstruction works: the decoder recovers a recognisable digit from the
32-dimensional bottleneck. But nothing in the training loss ever looked at
*where in $\mathbb{R}^{32}$* the encoder places its bottleneck codes for
real digits — only at whether the decoder can invert whatever it produces.
Sampling from a plausible-looking latent distribution and decoding exposes
exactly this gap.

In [ ]:
encoded_h, _ = ae_forward(ae_params, X)  # (32, N): where real digits actually land
print("encoder output range across training data:", encoded_h.min(), "to", encoded_h.max())
print("encoder output per-dimension std (first 8 dims):", encoded_h[:8].std(axis=1).round(2))

g = np.random.default_rng(SEED)
random_codes = g.uniform(-1, 1, (H, 8))  # matches tanh's (-1, 1) range, but not its true occupied shape
fake_images = ae_params["W2"] @ random_codes + ae_params["b2"]

fig, axes = plt.subplots(1, 8, figsize=(13, 2))
for ax, i in zip(axes, range(8)):
    ax.imshow(fake_images[:, i].reshape(28, 28), cmap="gray")
    ax.axis("off")
plt.suptitle("Decoding uniform-random bottleneck codes (range-matched, distribution-unmatched)")
plt.show()

These are not digits, even though every sampled code respects the
bottleneck's numerical range. **The autoencoder is not generative** because
its training objective never constrains the *shape* of the encoder's output
distribution — only whether the decoder can reconstruct from wherever real
data happens to land. A VAE fixes exactly this, by adding a term to the loss
that pulls the encoder's output distribution toward one we know how to
sample from.

## The Variational Autoencoder

Goal: maximise $\log p_\theta(x) = \log \int p_\theta(x \mid z)\, p(z)\, dz$ —
intractable, because it integrates over every possible $z$. Introduce an
encoder distribution $q_\phi(z \mid x)$ ("variational posterior") to turn
the integral into an expectation, then apply Jensen's inequality
($\log \mathbb{E}[\cdot] \geq \mathbb{E}[\log(\cdot)]$ for the concave $\log$):

$$\log p_\theta(x) = \log \mathbb{E}_{q_\phi(z|x)}\!\left[\frac{p_\theta(x,z)}{q_\phi(z|x)}\right]
\;\geq\; \mathbb{E}_{q_\phi(z|x)}\!\left[\log \frac{p_\theta(x,z)}{q_\phi(z|x)}\right]
= \underbrace{\mathbb{E}_{q_\phi(z|x)}[\log p_\theta(x|z)]}_{\text{reconstruction}}
- \underbrace{D_{KL}\big(q_\phi(z|x) \,\|\, p(z)\big)}_{\text{regulariser}}
\;=\; \text{ELBO}(x).$$

This lower bound on $\log p_\theta(x)$ — the **evidence lower bound (ELBO)** —
is what a VAE actually maximises, because the true log-likelihood cannot be
computed directly. Both terms have a job: reconstruction pushes $q_\phi$
toward codes the decoder can invert (the autoencoder's whole loss), and the
KL term is new — it pulls $q_\phi(z\mid x)$ toward a fixed, known prior
$p(z) = \mathcal N(0, I)$ for *every* $x$, which is exactly the missing
ingredient the autoencoder's random-code experiment showed was absent:
a latent distribution we know, in advance, how to sample from.

Choosing $q_\phi(z\mid x) = \mathcal N(\mu_\phi(x), \text{diag}(\sigma_\phi(x)^2))$
and $p(z) = \mathcal N(0, I)$ gives a closed-form KL term — no sampling needed
to evaluate it:

$$D_{KL}\big(\mathcal N(\mu, \sigma^2) \,\|\, \mathcal N(0, 1)\big)
= \tfrac12 \sum_j \left(\mu_j^2 + \sigma_j^2 - \log \sigma_j^2 - 1\right).$$

A quick numerical check confirms the closed form against a direct
Monte Carlo estimate of the same KL divergence for an arbitrary $(\mu,
\sigma)$.

In [ ]:
def kl_gaussian_closed_form(mu, sigma):
    return 0.5 * np.sum(mu ** 2 + sigma ** 2 - np.log(sigma ** 2) - 1)


def kl_gaussian_monte_carlo(mu, sigma, n_samples=2_000_000, seed=SEED):
    g = np.random.default_rng(seed)
    z = mu + sigma * g.normal(size=(n_samples, len(mu)))
    log_q = -0.5 * np.sum(((z - mu) / sigma) ** 2 + np.log(2 * np.pi * sigma ** 2), axis=1)
    log_p = -0.5 * np.sum(z ** 2 + np.log(2 * np.pi), axis=1)
    return float(np.mean(log_q - log_p))


mu_test, sigma_test = np.array([0.5, -1.2, 0.3]), np.array([0.8, 1.5, 0.4])
closed_form = kl_gaussian_closed_form(mu_test, sigma_test)
monte_carlo = kl_gaussian_monte_carlo(mu_test, sigma_test)
print(f"closed-form KL:   {closed_form:.4f}")
print(f"Monte Carlo KL:   {monte_carlo:.4f}")
assert abs(closed_form - monte_carlo) < 0.01, "closed-form Gaussian KL disagrees with a Monte Carlo estimate" 

## The Reparameterisation Trick

Training needs $\nabla_\phi \mathbb{E}_{q_\phi(z|x)}[\log p_\theta(x|z)]$, but
$z$ is *sampled* from a distribution whose own parameters ($\mu_\phi,
\sigma_\phi$) are what gradients must flow into — and sampling is not a
differentiable operation. The **reparameterisation trick** rewrites the
sample as a deterministic, differentiable function of $\phi$ and an
independent noise source:

$$z = \mu_\phi(x) + \sigma_\phi(x) \odot \epsilon, \qquad \epsilon \sim \mathcal N(0, I).$$

$z$ still has exactly the distribution $q_\phi(z\mid x)$ requires — only
where the randomness lives has moved, from a non-differentiable sampling
step to a fixed, external $\epsilon$ that ordinary backpropagation can see
straight through.

The alternative, gradient-estimator-free approach (REINFORCE / the
score-function estimator, $\nabla_\phi \mathbb{E}_{q_\phi}[f(z)] =
\mathbb{E}_{q_\phi}[f(z) \nabla_\phi \log q_\phi(z)]$) is unbiased too, but
empirically far noisier. A toy 1-D problem — minimise $\mathbb{E}_{z \sim
\mathcal N(\mu, 1)}[z^2]$ over $\mu$, whose true minimum is $\mu = 0$ — makes
the variance gap directly measurable.

In [ ]:
def reparam_grad_estimate(mu, n_samples, seed):
    g = np.random.default_rng(seed)
    eps = g.normal(size=n_samples)
    z = mu + eps           # reparameterised sample
    # f(z) = z^2, dz/dmu = 1 (mu only shifts the sample) -> df/dmu = 2z * 1
    return np.mean(2 * z)


def score_function_grad_estimate(mu, n_samples, seed):
    g = np.random.default_rng(seed)
    z = g.normal(mu, 1, size=n_samples)          # direct (non-reparameterised) sample
    f = z ** 2
    grad_log_q = z - mu                           # d/dmu log N(z; mu, 1) = z - mu
    return np.mean(f * grad_log_q)


TRUE_GRAD = 2 * 1.5  # at mu=1.5, d/dmu E[z^2] = 2*mu exactly

reparam_estimates = [reparam_grad_estimate(1.5, 20, seed) for seed in range(500)]
score_estimates = [score_function_grad_estimate(1.5, 20, seed) for seed in range(500)]

print(f"true gradient at mu=1.5:                {TRUE_GRAD:.3f}")
print(f"reparameterised estimator  -- mean: {np.mean(reparam_estimates):.3f}, std: {np.std(reparam_estimates):.3f}")
print(f"score-function estimator   -- mean: {np.mean(score_estimates):.3f}, std: {np.std(score_estimates):.3f}")

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(reparam_estimates, bins=30, alpha=0.6, label="reparameterisation")
ax.hist(score_estimates, bins=30, alpha=0.6, label="score function")
ax.axvline(TRUE_GRAD, color="k", linestyle="--", label="true gradient")
ax.set_xlabel("gradient estimate"); ax.set_ylabel("count"); ax.legend()
ax.set_title("Same expectation, same 20 samples/estimate -- very different variance")
plt.tight_layout(); plt.show()
assert np.std(reparam_estimates) < np.std(score_estimates), \
    "reparameterisation should be the lower-variance estimator here" 

Both estimators are unbiased — both histograms centre on the true gradient —
but the score-function estimator's spread is dramatically wider for the same
sample budget. This variance gap, not just differentiability in principle,
is why every practical VAE uses reparameterisation rather than
score-function gradients to train its encoder.

## Diffusion Models

A VAE's encoder compresses to a single latent step; diffusion instead
destroys the image gradually over many small steps, then learns to reverse
that destruction. The **forward process** adds a little Gaussian noise at
each of $T$ steps:

$$x_t = \sqrt{1 - \beta_t}\, x_{t-1} + \sqrt{\beta_t}\, \epsilon_t, \qquad \epsilon_t \sim \mathcal N(0, I),$$

with a small noise schedule $\beta_t$ chosen so each step barely perturbs the
image. Writing $\alpha_t = 1-\beta_t$ and unrolling the recursion (each step
is itself Gaussian, and a sum of independent Gaussians is Gaussian) gives a
direct, closed-form jump from $x_0$ to *any* $x_t$ without simulating the
intermediate steps:

$$x_t = \sqrt{\bar\alpha_t}\, x_0 + \sqrt{1 - \bar\alpha_t}\, \epsilon,
\qquad \bar\alpha_t = \prod_{s=1}^{t} \alpha_s, \qquad \epsilon \sim \mathcal N(0, I).$$

In [ ]:
T = 200
betas = np.linspace(1e-4, 0.02, T)
alphas = 1.0 - betas
alpha_bars = np.cumprod(alphas)


def forward_noise(x0, t, seed):
    g = np.random.default_rng(seed)
    eps = g.normal(size=x0.shape)
    return np.sqrt(alpha_bars[t]) * x0 + np.sqrt(1 - alpha_bars[t]) * eps


x0 = X[:, 0]  # one MNIST digit, flattened, pixels in [0, 1]
t_steps = [0, 5, 20, 50, 100, 199]
fig, axes = plt.subplots(1, len(t_steps), figsize=(15, 2.5))
for ax, t in zip(axes, t_steps):
    xt = forward_noise(x0, t, seed=SEED) if t > 0 else x0
    ax.imshow(xt.reshape(28, 28), cmap="gray")
    ax.set_title(f"t={t}\n$\\bar\\alpha_t$={alpha_bars[t]:.3f}" if t > 0 else "t=0 (original)", fontsize=9)
    ax.axis("off")
plt.suptitle("Forward noising: closed-form $x_t$ from $x_0$, no intermediate steps simulated")
plt.tight_layout(); plt.show()

The schedule is designed so that by $t=T$, $\bar\alpha_T \approx 0$ and
$x_T \approx \epsilon$ — pure noise, entirely independent of $x_0$. A
direct check confirms this numerically: sampling many $x_T$ values from many
different starting digits should look statistically identical to sampling
$\epsilon \sim \mathcal N(0, I)$ directly.

In [ ]:
x_T_samples = np.stack([forward_noise(X[:, i], T - 1, seed=i) for i in range(200)])
pure_noise_samples = np.random.default_rng(SEED).normal(size=x_T_samples.shape)

print(f"alpha_bar at t=T-1: {alpha_bars[-1]:.2e}  (should be close to 0)")
print(f"x_T   -- mean: {x_T_samples.mean():.3f}, std: {x_T_samples.std():.3f}")
print(f"noise -- mean: {pure_noise_samples.mean():.3f}, std: {pure_noise_samples.std():.3f}")

The **reverse process** is what makes diffusion generative: start from pure
noise $x_T \sim \mathcal N(0, I)$ and learn a network $p_\theta(x_{t-1} \mid
x_t)$ that undoes one noising step at a time, eventually producing a clean
$x_0$. Exactly as in the VAE, the training objective comes from an ELBO —
here on a chain of $T$ latent variables instead of one — and after the
standard reweighting (Ho et al., 2020) it collapses to a strikingly simple
form: predict the noise $\epsilon$ that was added, given the noisy image and
the timestep:

$$L_{\text{simple}}(\theta) = \mathbb{E}_{x_0, t, \epsilon}\left[
\left\lVert \epsilon - \epsilon_\theta(x_t, t) \right\rVert^2\right],
\qquad x_t = \sqrt{\bar\alpha_t}\, x_0 + \sqrt{1-\bar\alpha_t}\, \epsilon.$$

This is exactly the VAE's reconstruction term in a different guise: instead
of reconstructing $x$ from a compressed $z$, the network reconstructs the
*noise* that was added at a random step, and doing this correctly at every
noise level is what lets sampling walk backward from $x_T$ to a clean image
one small, learnable step at a time.

## Comparing Generative Families

A generative adversarial network (GAN) optimises neither an ELBO nor a
denoising objective — it pits a generator against a discriminator in a
minimax game with no explicit likelihood term at all:

$$\min_G \max_D \; \mathbb{E}_{x \sim p_{\text{data}}}[\log D(x)]
+ \mathbb{E}_{z \sim p(z)}[\log(1 - D(G(z)))].$$

$D$ learns to tell real data from $G$'s output; $G$ learns to fool $D$. There
is no term anywhere in this objective that measures how likely the training
data is under $G$ — sample quality is judged entirely by whether a learned
critic can be fooled, not by a tractable probability.

In [ ]:
import pandas as pd
comparison = pd.DataFrame([
    {"Family": "Autoencoder", "Optimises": "Reconstruction error only",
     "Likelihood": "None (no generative guarantee)", "Sampling": "Undefined — no known latent distribution"},
    {"Family": "VAE", "Optimises": "ELBO = reconstruction - KL(q||p)",
     "Likelihood": "Lower-bounded, tractable", "Sampling": "z ~ N(0,I), single decode step"},
    {"Family": "Diffusion", "Optimises": "Denoising MSE (a reweighted ELBO)",
     "Likelihood": "Lower-bounded, tractable", "Sampling": "x_T ~ N(0,I), T iterative denoise steps"},
    {"Family": "GAN", "Optimises": "Adversarial minimax game",
     "Likelihood": "None (no explicit density)", "Sampling": "z ~ p(z), single generate step"},
])
comparison

The practical consequences track the objective directly: a VAE's explicit
likelihood bound makes training stable but its single-step decode tends to
blur fine detail; diffusion's many small, well-defined denoising steps
produce sharper samples at the cost of many forward passes per sample; a
GAN's adversarial game can produce the sharpest single-step samples but the
minimax objective has no convergence guarantee and is notoriously prone to
instability (mode collapse) — a direct consequence of optimising a game
rather than a bound on a real quantity.

## Key Takeaways

- A plain **autoencoder** minimises reconstruction error only, and its
  encoder's output distribution is unconstrained — confirmed here by
  decoding random bottleneck codes into non-digit noise despite matching the
  bottleneck's numeric range.
- The **VAE** fixes this by maximising the **ELBO** (reconstruction minus a
  KL term pulling the latent toward a known, sampleable prior), derived here
  from Jensen's inequality, with the Gaussian-Gaussian KL's closed form
  confirmed against a direct Monte Carlo estimate.
- The **reparameterisation trick** rewrites a stochastic sample as a
  differentiable function of the parameters plus external noise — measured
  here as dramatically lower gradient-estimator variance than the
  alternative score-function estimator at an identical sample budget.
- **Diffusion** replaces one latent step with many small ones; its forward
  process has an exact closed form ($x_t$ from $x_0$ directly, confirmed
  numerically to reach $\mathcal N(0, I)$ by $t=T$), and its reverse training
  objective is a reweighted ELBO that collapses to simple noise prediction.
- **GANs** optimise neither bound — an adversarial minimax game instead —
  trading likelihood tractability and training stability for often sharper
  single-step samples.